# Class 3 (bonus) — RAG with LangChain & Vector Databases

**Week 6: Foundations of RAG and Chatbots** · companion to `class-3.ipynb`

The main Class 3 notebook builds RAG **from scratch** (a Python list of vectors). This notebook is the same pipeline with **LangChain** and real vector databases.

You will:
- Split the same style of policy docs into chunks
- Embed locally (`all-MiniLM-L6-v2` — Groq has no public embedding model)
- Store vectors in **Chroma**, retrieve, then generate with Groq
- Swap the same chunks into **FAISS** (same job, different vector DB)
- Optionally attach the retriever as a LangChain **tool** (the Week 5 agent pattern, now over documents)

Run cells in order with **Shift+Enter**. Generation cells need a `GROQ_API_KEY` (Colab secret or env var). Indexing and retrieval run without a key.

## Setup

```bash
export GROQ_API_KEY="gsk-..."
```

In Colab: add a secret named `GROQ_API_KEY` and enable notebook access.

In [1]:
!pip install -q langchain langchain-groq langchain-chroma langchain-huggingface langchain-text-splitters langchain-community sentence-transformers faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 

In [2]:
import os

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
except Exception:
    pass

if not GROQ_API_KEY:
    print(
        "No GROQ_API_KEY found.\n"
        "Set it in your environment or add a Colab secret named GROQ_API_KEY.\n"
        "Indexing and retrieval still run. Generation cells will skip."
    )
else:
    print("Found GROQ_API_KEY. RAG generation cells are ready.")

Found GROQ_API_KEY. RAG generation cells are ready.


## 1. The same policy docs as Class 3

These three policies match the from-scratch Class 3 notebook, so you can compare a Python list of vectors with Chroma and FAISS.

In [3]:
from langchain_core.documents import Document

raw_documents = [
    Document(
        page_content=(
            "Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting from their first day. "
            "Unused days roll over up to a maximum of 5 days into the next year."
        ),
        metadata={"source": "vacation.md"},
    ),
    Document(
        page_content=(
            "Expense Policy: Employees must submit expense reports within 30 days of purchase using the finance portal. "
            "Reports missing a receipt over $25 will be returned for correction."
        ),
        metadata={"source": "expenses.md"},
    ),
    Document(
        page_content=(
            "Remote Work Policy: Remote employees must be reachable by chat or phone between 10am and 3pm in their local time zone, "
            "and are expected to attend the weekly all-hands meeting on video."
        ),
        metadata={"source": "remote.md"},
    ),
]

for d in raw_documents:
    print(f"[{d.metadata['source']}] {d.page_content[:88]}...")

[vacation.md] Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly start...
[expenses.md] Expense Policy: Employees must submit expense reports within 30 days of purchase using t...
[remote.md] Remote Work Policy: Remote employees must be reachable by chat or phone between 10am and...


## 2. Chunk the documents

LangChain's `RecursiveCharacterTextSplitter` is the usual first splitter: it tries paragraphs, then sentences, then characters. Our docs are short, so most stay as a single chunk — the same API still works when files get long.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=280, chunk_overlap=40)
chunks = splitter.split_documents(raw_documents)
print(f"{len(chunks)} chunk(s):")
for c in chunks:
    print(f"  [{c.metadata['source']}] {c.page_content}")

3 chunk(s):
  [vacation.md] Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting from their first day. Unused days roll over up to a maximum of 5 days into the next year.
  [expenses.md] Expense Policy: Employees must submit expense reports within 30 days of purchase using the finance portal. Reports missing a receipt over $25 will be returned for correction.
  [remote.md] Remote Work Policy: Remote employees must be reachable by chat or phone between 10am and 3pm in their local time zone, and are expected to attend the weekly all-hands meeting on video.


## 3. Embed and store in Chroma

Chroma is an open-source vector database that runs locally — a good prototype default (Week 6 also covers FAISS and Pinecone). Embeddings run on your machine with MiniLM; no embedding API key.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="loop-labs-handbook",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"Indexed {len(chunks)} chunk(s) into Chroma.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 3 chunk(s) into Chroma.


## 4. Retrieve (no LLM yet)

Ask a question in plain language. Chroma returns the nearest chunks — the same semantic search you ranked by hand in Class 2.

In [6]:
question = "How many vacation days do new hires get?"
hits = retriever.invoke(question)
for i, doc in enumerate(hits, 1):
    print(f"{i}. [{doc.metadata['source']}] {doc.page_content}")

1. [vacation.md] Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting from their first day. Unused days roll over up to a maximum of 5 days into the next year.
2. [expenses.md] Expense Policy: Employees must submit expense reports within 30 days of purchase using the finance portal. Reports missing a receipt over $25 will be returned for correction.


## 5. Generate a grounded answer with Groq

Stuff the retrieved chunks into a prompt and ask Groq (`llama-3.3-70b-versatile`) to answer **only** from that context. That retrieve → prompt → generate loop is RAG.

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_template(
    """Answer using ONLY the context below. If the answer is not in the context, say you don't know.

CONTEXT:
{context}

QUESTION: {question}
"""
)


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


def make_rag():
    if not GROQ_API_KEY:
        print("Skipping RAG chain — no GROQ_API_KEY set.")
        return None
    llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, api_key=GROQ_API_KEY)
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )


rag = make_rag()
if rag is not None:
    print(rag.invoke(question))

New hires accrue **15 paid vacation days per year**.


Try a question the handbook does **not** cover — a good RAG prompt should refuse to invent an answer.

In [9]:
if rag is None:
    print("Skipping — no GROQ_API_KEY set.")
else:
    print(rag.invoke("What is the pet-friendly office stipend?"))

I don't know.


## 6. Same chunks, different vector DB: FAISS

Chroma, FAISS, and Pinecone all store vectors and return nearest neighbors. Swap the store; keep the embeddings and the RAG prompt. FAISS is a local similarity-search library (you manage persistence). Pinecone is the hosted option — it needs its own API key, so we skip it here.

In [11]:
from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(chunks, embeddings)
faiss_retriever = faiss_store.as_retriever(search_kwargs={"k": 2})

print("FAISS hits for the same PTO question:")
for i, doc in enumerate(faiss_retriever.invoke(question), 1):
    print(f"  {i}. [{doc.metadata['source']}] {doc.page_content[:90]}...")

# Point the RAG chain at FAISS instead of Chroma (same prompt + Groq).
if GROQ_API_KEY:
    llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, api_key=GROQ_API_KEY)
    faiss_rag = (
        {"context": faiss_retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("\nFAISS-grounded answer:")
    print(faiss_rag.invoke(question))
else:
    print("\nSkipping FAISS generation — no GROQ_API_KEY set.")

FAISS hits for the same PTO question:
  1. [vacation.md] Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly startin...
  2. [expenses.md] Expense Policy: Employees must submit expense reports within 30 days of purchase using the...

FAISS-grounded answer:
New hires accrue **15 paid vacation days per year**.


## 7. Retriever as an agent tool (optional)

Week 5's agent used tools like `calculator` and `mock_search`. A handbook retriever is the same pattern: a tool the model can call when the question is about policy. Chat memory (the message list) and document memory (the vector DB) solve different jobs — you often want both.

In [12]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook. Use for vacation, expenses, and remote-work questions."""
    docs = retriever.invoke(query)
    if not docs:
        return "No handbook passages found."
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


def ask_handbook_agent(text: str):
    if not GROQ_API_KEY:
        print("Skipping agent — no GROQ_API_KEY set.")
        return None
    llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, api_key=GROQ_API_KEY)
    agent = create_agent(
        model=llm,
        tools=[search_handbook],
        system_prompt=(
            "You are the employee handbook assistant. "
            "Call search_handbook before answering policy questions. "
            "If the tool result does not contain the answer, say you don't know."
        ),
    )
    result = agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config={"recursion_limit": 8},
    )
    content = result["messages"][-1].content
    print(content)
    return result


ask_handbook_agent("How many vacation days do new hires get?")

New hires accrue **15 paid vacation days per year**, credited monthly starting from their first day.


{'messages': [HumanMessage(content='How many vacation days do new hires get?', additional_kwargs={}, response_metadata={}, id='72983d90-58bc-49a2-9f0f-d461b0aadc35'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call search_handbook with query about vacation days for new hires.', 'tool_calls': [{'id': 'fc_ec7c2fb3-d75b-46ca-95a7-c4edd7604483', 'function': {'arguments': '{"query":"vacation days new hires"}', 'name': 'search_handbook'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 173, 'total_tokens': 222, 'completion_time': 0.103310577, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.006594827, 'prompt_tokens_details': None, 'queue_time': 0.023992702, 'total_time': 0.109905404}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_df9620fe21', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a01d80-d

## Challenges

Each one builds on `chunks`, `embeddings`, `retriever`, and `rag` above.

### Challenge 1 — Add a fourth document

Add a short equipment policy (e.g. company laptops), re-run the splitter + Chroma index, and ask whether laptops are provided.

In [ ]:
# TODO: append a Document, rebuild vectorstore/retriever, ask a remote-work question

In [14]:
raw_documents.append(
    Document(
        page_content="The government provides digital identity services to citizens. Citizens can access selected government services online using their digital identity."
    )
)

# Re-run the splitter
chunks = splitter.split_documents(raw_documents)

# Rebuild the Chroma vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Rebuild the retriever
retriever = vectorstore.as_retriever()

# Ask a question about the government policy
question = "Does the government provide digital identity services to citizens?"

# Retrieve relevant information
results = retriever.invoke(question)

# Display the result
for result in results:
    print(result.page_content)

The government provides digital identity services to citizens. Citizens can access selected government services online using their digital identity.
Remote Work Policy: Remote employees must be reachable by chat or phone between 10am and 3pm in their local time zone, and are expected to attend the weekly all-hands meeting on video.
Expense Policy: Employees must submit expense reports within 30 days of purchase using the finance portal. Reports missing a receipt over $25 will be returned for correction.
Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting from their first day. Unused days roll over up to a maximum of 5 days into the next year.


### Challenge 2 — Force a "don't know"

Ask `rag` something the three original documents never mention. Confirm the model refuses to guess.

In [ ]:
# TODO: rag.invoke(...) on an uncovered question

### Challenge 3 — Change k

Rebuild the retriever with `k=1` and `k=3`. Print the retrieved sources for the PTO question and note what extra (or missing) context does to the answer.

In [ ]:
# TODO: compare retriever search_kwargs k=1 vs k=3

### Challenge 4 (stretch) — Cite the source

Change the prompt so the answer includes the `source` filename inline (e.g. `(vacation.md)`).

In [ ]:
# TODO: update the prompt / format_docs and re-run rag.invoke